In [1]:
import os
import re
import time
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed


In [2]:
INPUT_DIR = "/home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election"

def find_all_files_recursively(directory, extension=".csv.gz"):
    files = []
    for root, _, filenames in os.walk(directory):
        for filename in filenames:
            if filename.endswith(extension):
                files.append(os.path.join(root, filename))
    return files

In [9]:
def inspect_location_data(file_path):
    try:
        header = pd.read_csv(
            file_path,
            compression="gzip",
            nrows=0
        ).columns

        has_location = "location" in header
        has_lang = "lang" in header

        if not has_location or not has_lang:
            return {
                "file": file_path,
                "has_location_column": False,
                "total_rows": 0,
                "location_present": 0
            }

        df = pd.read_csv(
            file_path,
            compression="gzip",
            usecols=["location", "lang"],
            dtype=str,
            low_memory=False
        )

        df = df[df["lang"] == "en"]

        return {
            "file": file_path,
            "has_location_column": True,
            "total_rows": len(df),
            "location_present": df["location"].notna().sum(),
            "location_missing": df["location"].isna().sum()
        }

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None



In [10]:
print(INPUT_DIR)

/home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election


In [11]:
files = find_all_files_recursively(INPUT_DIR)
print(f"Found {len(files)} files")

summaries = []
for i, file in enumerate(files):
    result = inspect_location_data(file)
    if result:
        summaries.append(result)

    if i % 50 == 0:
        print(f"Processed {i}/{len(files)} files")

print(len(summaries))

Found 881 files
Processed 0/881 files
Processed 50/881 files
Processed 100/881 files
Processed 150/881 files
Processed 200/881 files
Processed 250/881 files
Processed 300/881 files
Processed 350/881 files
Processed 400/881 files
Processed 450/881 files
Processed 500/881 files
Processed 550/881 files
Processed 600/881 files
Processed 650/881 files
Processed 700/881 files
Processed 750/881 files
Processed 800/881 files
Processed 850/881 files
881


In [19]:
summary_df = pd.DataFrame(summaries)

print(summary_df[["total_rows", "location_present", "location_missing"]].describe())

# Overall availability
total_rows = summary_df["total_rows"].sum()
total_locations = summary_df["location_present"].sum()

print(f"\nOverall rows: {total_rows}")
print(f"Rows with location data: {total_locations}")
print(f"Location availability: {total_locations / total_rows:.2%}")


         total_rows  location_present  location_missing
count    881.000000        881.000000        834.000000
mean   39533.811578       1734.544835      39929.441247
std    12287.100187       7365.444215      10820.035593
min        0.000000          0.000000          0.000000
25%    41929.000000          0.000000      41686.250000
50%    43598.000000          0.000000      43542.000000
75%    44813.000000          0.000000      44777.250000
max    47160.000000      45720.000000      47160.000000

Overall rows: 34829288
Rows with location data: 1528134
Location availability: 4.39%


In [22]:
summary_df[summary_df['location_present'] > 0]

,file,has_location_column,total_rows,location_present,location_missing
4,/home/tsaddi01/msc-project-source-code-files-2...,True,43342,12928,30414.0
10,/home/tsaddi01/msc-project-source-code-files-2...,True,45899,3347,42552.0
87,/home/tsaddi01/msc-project-source-code-files-2...,True,44354,513,43841.0
173,/home/tsaddi01/msc-project-source-code-files-2...,True,45595,5813,39782.0
176,/home/tsaddi01/msc-project-source-code-files-2...,True,43001,25765,17236.0
...,...,...,...,...,...
753,/home/tsaddi01/msc-project-source-code-files-2...,True,42088,25858,16230.0
774,/home/tsaddi01/msc-project-source-code-files-2...,True,45588,13200,32388.0
792,/home/tsaddi01/msc-project-source-code-files-2...,True,43727,21277,22450.0
804,/home/tsaddi01/msc-project-source-code-files-2...,True,40991,32401,8590.0


In [3]:
CAMPAIGN_START = pd.Timestamp("2024-05-01")

def check_file_for_pre_campaign(file_path):
    """
    Check a single CSV/GZ file for tweets before campaign start.
    
    Returns:
        tuple(file_path, pre_count, min_ts, max_ts) if pre-campaign tweets exist, else None
    """
    try:
        df = pd.read_csv(file_path, usecols=["date", "epoch"], dtype={"date": str, "epoch": float})
        
        # Convert date to datetime
        df['timestamp'] = pd.to_datetime(df['date'], errors='coerce')
        
        pre_mask = df['timestamp'] < CAMPAIGN_START
        pre_count = pre_mask.sum()
        if pre_count > 0:
            min_ts = df['timestamp'].min()
            max_ts = df['timestamp'].max()
            return (file_path, pre_count, min_ts, max_ts)
    except Exception as e:
        return (file_path, f"Error: {e}")
    
    return None


all_files = find_all_files_recursively(INPUT_DIR, extension=".csv.gz")
print(f"Total files found: {len(all_files)}")

files_with_early_dates = []

# Parallel processing for speed
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    futures = {executor.submit(check_file_for_pre_campaign, f): f for f in all_files}
    for future in as_completed(futures):
        result = future.result()
        if result:
            files_with_early_dates.append(result)

print(f"Files containing pre-campaign tweets: {len(files_with_early_dates)}")

# Save summary CSV
summary_df = pd.DataFrame(
    files_with_early_dates,
    columns=["file_path", "pre_campaign_count", "min_timestamp", "max_timestamp"]
)


Total files found: 881
Files containing pre-campaign tweets: 33


In [12]:
total_pre_campaign = summary_df['pre_campaign_count'].sum()
print("Total pre-campaign tweets:", total_pre_campaign)

Total pre-campaign tweets: 279941


In [6]:
pd.set_option('display.max_rows', None)       
pd.set_option('display.max_colwidth', None) 

print(summary_df['file_path'])


0      /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_8/may_july_chunk_153.csv.gz
1     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_19/may_july_chunk_362.csv.gz
2     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_19/may_july_chunk_379.csv.gz
3     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_21/may_july_chunk_420.csv.gz
4     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_21/may_july_chunk_419.csv.gz
5     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_21/may_july_chunk_418.csv.gz
6     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_19/may_july_chunk_380.csv.gz
7     /home/tsaddi01/msc-project-source-code-files-24-25-tayyib-saddique/x-24-us-election/part_22/may_july_chunk_436.csv.gz
8     /h

In [17]:
import pandas as pd
df = pd.read_parquet('train_unlabelled.parquet')

In [20]:
df['timestamp'].min()

Timestamp('2009-10-13 00:00:00')